In [1]:
import pandas as pd
import sqlite3

# Loading Raw ACS Data into a DataFrame

In [2]:
df = pd.read_excel("../data/oo_raw.xlsx")

# Exploratory Data Analysis

Using a Custom Function from the Utilities Notebook

In [3]:
from utilities import basic_eda

basic_eda(df)

DataFrame Shape: (5245, 23)

 Column Names:
['Area Name', 'Area Type', 'SOC Title', 'Standard Occupational Classification (SOC)', 'SOC Major Group', 'SOC Classification', '2022 Estimated Employment', '2032 Projected Employment', 'Change', 'Percent Change', 'Annualized Percent Growth (Grow Rate)', 'Exits', 'Transfers', 'Openings', 'Mean Annual', 'Entry Annual', '25th Percentile Anual', 'Median Annual', '75th Percentile Annual', 'Experienced Annual', 'Typical Education Required for Entry', 'Typical Work Experience Required in Related Occupation', 'Typical On-the-Job Training Required to Achieve Competency']

 Data Types:
Area Name                                                      object
Area Type                                                      object
SOC Title                                                      object
Standard Occupational Classification (SOC)                      int64
SOC Major Group                                                 int64
SOC Classification     

# Cleaning

## Column Names

I will start by changing column names to descriptive ones in lower snake case.

In [135]:
# Create a column renaming dictionary
column_rename_map = {
    "Area Name": "area_name",
    "Area Type": "area_type",
    "SOC Title": "soc_title",
    "Standard Occupational Classification (SOC)": "occupation",
    "SOC Major Group": "soc_major_group",
    "SOC Classification": "soc_classification",
    "2022 Estimated Employment": "employment_2022",
    "2032 Projected Employment": "employment_2032",
    "Change": "employment_change",
    "Percent Change": "percent_change",
    "Annualized Percent Growth (Grow Rate)": "annualized_percent_growth",
    "Exits": "exits",
    "Transfers": "transfers",
    "Openings": "openings",
    "Mean Annual": "mean_annual_wage",
    "Entry Annual": "entry_annual_wage",
    "25th Percentile Anual": "percentile_25_wage",
    "Median Annual": "median_annual_wage",
    "75th Percentile Annual": "percentile_75_wage",
    "Experienced Annual": "experienced_annual_wage",
    "Typical Education Required for Entry": "education_required",
    "Typical Work Experience Required in Related Occupation": "work_experience_required",
    "Typical On-the-Job Training Required to Achieve Competency": "ojt_required"
}

# Apply the renaming
df = df.rename(columns=column_rename_map)

# Check the result
print(df.columns.tolist())


['area_name', 'area_type', 'soc_title', 'occupation', 'soc_major_group', 'soc_classification', 'employment_2022', 'employment_2032', 'employment_change', 'percent_change', 'annualized_percent_growth', 'exits', 'transfers', 'openings', 'mean_annual_wage', 'entry_annual_wage', 'percentile_25_wage', 'median_annual_wage', 'percentile_75_wage', 'experienced_annual_wage', 'education_required', 'work_experience_required', 'ojt_required']


## Handling Nulls

### Observations
1. The columns 'soc_classification', 'work_experience_required', and 'ojt_required' all have a large percentage of nulls.  
2. Several other columns have low percentages of nulls.

### Thoughts/Plan
1. Eliminate the 3 columns with nulls >10%. They are not necessary.
2. Fill nulls in the 'education_required' column to 'Not reported'.
3. Leave other nulls in Dataframe. 
    - These columns have null percentages <10/%.
    - Using a mean or median fill value for these columns would be misleading.
    - Nulls can be filtered during analysis.

In [136]:
columns_to_drop = [
    "soc_classification",
    "work_experience_required",
    "ojt_required"
]

df = df.drop(columns=columns_to_drop) # Eliminating columns with high percentage of nulls

df["education_required"] = df["education_required"].fillna("Not reported") # Replacing nulls in this column with "Not reported"


## Adding a Column
Using Census Occupation Code List Crosswalk  

Using the 'soc' column values, I will create a 'census_job_title' column. This will ensure occupations listed match the wording for occupations listed in my other data source, since they use different codes.

In [137]:
# Reformating 'occupation' column to match '11-1011' format
df["occupation"] = df["occupation"].astype(str)
df["occupation"] = df["occupation"].str[:2] + "-" + df["occupation"].str[2:]

# Loading the crosswalk Excel file
crosswalk_df = pd.read_excel("../docs/job_code_crosswalk.xlsx", sheet_name="NEM SOC ACS crosswalk", header=4)

# Creating the mapping dictionary
crosswalk_df["Matrix Occupation Code"] = crosswalk_df["Matrix Occupation Code"].astype(str)
occupation_code_dictionary = dict(zip(crosswalk_df["Matrix Occupation Code"], crosswalk_df["ACS Occupational Title"]))

# Mapping titles in place using the dictionary
df["occupation"] = df["occupation"].map(occupation_code_dictionary)

df['occupation'].head()

0                                    NaN
1                                    NaN
2       Chief executives and legislators
3        General and operations managers
4    Advertising and promotions managers
Name: occupation, dtype: object

Thoughts on output: The column originally had no null values, so this doesn't look correct.

Checking total number of nulls:

In [138]:
df['occupation'].isnull().sum()

np.int64(253)

Displaying a sample of rows with null 'occupation' values:

In [139]:
df[df["occupation"].isna()].head(10)


,area_name,area_type,soc_title,occupation,soc_major_group,employment_2022,employment_2032,employment_change,percent_change,annualized_percent_growth,exits,transfers,openings,mean_annual_wage,entry_annual_wage,percentile_25_wage,median_annual_wage,percentile_75_wage,experienced_annual_wage,education_required
0,Kentucky,State Level,Total All occupations,NaN,0,2049528,2146969,97441,4.7543,0.4656,1005096,1288925,2391462,54030.0,24880.0,31600.0,43730.0,62060.0,89810.0,Not reported
1,Kentucky,State Level,Management Occupations,NaN,11,132055,142022,9967,7.5476,0.7303,38848,64583,113398,105980.0,43870.0,61360.0,91230.0,129120.0,176720.0,Not reported
36,Kentucky,State Level,Business and Financial Operations Occupations,NaN,13,93329,98528,5199,5.5706,0.5436,29220,45539,79958,73360.0,39130.0,49910.0,65000.0,85650.0,109520.0,Not reported
66,Kentucky,State Level,Computer and Mathematical Occupations,NaN,15,40424,45571,5147,12.7325,1.2057,9775,15855,30777,84790.0,43060.0,55890.0,79620.0,105490.0,133800.0,Not reported
85,Kentucky,State Level,Architecture and Engineering Occupations,NaN,17,28111,30376,2265,8.0573,0.7779,8559,11313,22137,81240.0,43700.0,57410.0,78330.0,100720.0,124850.0,Not reported
118,Kentucky,State Level,"Life, Physical, and Social Science Occupations",NaN,19,14307,15304,997,6.9686,0.6759,3270,10038,14305,69820.0,38740.0,47670.0,62700.0,82910.0,107990.0,Not reported
158,Kentucky,State Level,Community and Social Service Occupations,NaN,21,34780,38144,3364,9.6722,0.9275,13250,17162,33776,50230.0,31160.0,36470.0,45490.0,59910.0,77220.0,Not reported
176,Kentucky,State Level,Legal Occupations,NaN,23,11909,12457,548,4.6016,0.4509,3423,3589,7560,92460.0,40000.0,49550.0,70200.0,118620.0,167310.0,Not reported
181,Kentucky,State Level,Educational Instruction and Library Occupations,NaN,25,91922,93760,1838,1.9995,0.1982,40046,38378,80262,56440.0,28910.0,40140.0,51950.0,64390.0,83180.0,Not reported
244,Kentucky,State Level,"Arts, Design, Entertainment, Sports, and Media...",NaN,27,22205,22952,747,3.3641,0.3314,9014,12794,22555,53680.0,26410.0,31800.0,43570.0,61980.0,84720.0,Not reported


Conclusion: The rows that became null after mapping are likely headers for occupational groups and would not have had a mappable code.

Filling them with the values in the 'soc_title' column would prevent later table merging. 

Dropping these rows:

In [140]:
df = df[df["occupation"].notna()].copy()

## Rechecking Data Types

In [141]:
df.dtypes

area_name                     object
area_type                     object
soc_title                     object
occupation                    object
soc_major_group                int64
employment_2022                int64
employment_2032                int64
employment_change              int64
percent_change               float64
annualized_percent_growth    float64
exits                          int64
transfers                      int64
openings                       int64
mean_annual_wage             float64
entry_annual_wage            float64
percentile_25_wage           float64
median_annual_wage           float64
percentile_75_wage           float64
experienced_annual_wage      float64
education_required            object
dtype: object

# Converting to SQLite Database Table

In [142]:
# Connect to existing SQLite database
conn = sqlite3.connect("../data/cleaned_data.sqlite")

# Write the DataFrame `df` to the database as a new table called "puma_data"
df.to_sql(
    name="oo_data",       # name of the new table
    con=conn,               # database connection
    if_exists="replace",    # overwrite if table already exists
    index=False             # don't include the index as a column
)

# Close the connection
conn.close()

print("Table 'oo_data' successfully added to cleaned_data.sqlite.")

Table 'oo_data' successfully added to cleaned_data.sqlite.
